# **Bitcoin Cycle Risk Model (0-100)**

A single composite score that answers one question: **where are we in the cycle?** 0 = cycle lows (historically the best place to buy), 100 = cycle highs (historically the best place to sell).

## How it is built

Three indicators, each ranked against its own trailing **4-year** distribution (one halving cycle), then blended with backtest-selected weights:

| Component | Weight | What it measures |
|---|---|---|
| `pl_resid` | 0.60 | Deviation of price from its own expanding power-law trend (refit on every bar, no lookahead) |
| `mvrv` | 0.20 | MVRV ratio - on-chain valuation vs. aggregate cost basis |
| `mult_730` | 0.20 | Price / 730-day MA (the 2-Year MA Multiplier) |

Weights were chosen by backtest, not by taste: the power-law residual carries nearly all of the predictive power, while MVRV and the 2-year multiple keep the score from resting on a single indicator. Weights are renormalised over whatever components are available on a given day, so the score survives CoinMetrics' on-chain feed running months behind.

Two variants are produced: `risk` (all three components) and `risk_price` (price-only, always live, and the one the projection extends).

## Validation

`validate_risk_model` checks the only thing that matters: does a high reading today actually mean poor forward returns? It reports median forward return and win rate per risk bucket at 90/180/365 days, plus the Spearman rank correlation.

**Honest reading of the results:** the edge is real but concentrated at the extremes and grows with the horizon. The 80-100 bucket has historically produced *negative* median returns at every horizon, and the bottom buckets strongly positive ones - but the middle of the range carries little information, and at 90 days the correlation is weak (statistically significant, but small). This is a cycle-position tool, not a short-term timing signal.

## Projection: historical analogs

`project_risk` does **not** simulate random price paths. A block-bootstrap Monte Carlo was tried first and discarded: resampling returns ignores the mean reversion and cycle structure the score itself demonstrates, so it degenerates into a widening cone centred on today's value - technically correct, practically useless.

Instead the projection is conditional and empirical. It finds every past episode that started at **today's risk level and today's 30-day direction of travel**, then shows what the score actually did over the following 6 months in each of those episodes, together with the BTC price return that came with it.

**Caveat, and it matters:** this rests on a handful of overlapping episodes drawn from only three completed cycles. Read the spread as "here is what happened the last few times", not as a calibrated probability.

*Not financial advice. Past relationships can and do break down.*


In [ ]:
import sys
import sys, os
_base = os.getcwd()
while _base and not os.path.isdir(os.path.join(_base, 'functions')):
    _parent = os.path.dirname(_base)
    if _parent == _base: break
    _base = _parent
if os.path.abspath(_base) not in sys.path:
    sys.path.insert(0, os.path.abspath(_base))
from functions.functions_RiskModel import (
    load_dataframe,
    plot_risk_model,
    validate_risk_model,
    project_risk,
)
import warnings
warnings.filterwarnings("ignore")

df = load_dataframe()

## The risk score through history

In [ ]:
plot_risk_model(df)

## Validation: do high readings actually precede poor returns?

In [ ]:
validation = validate_risk_model(df)

## Projection: what followed the closest historical analogs?

In [ ]:
projection = project_risk(df, horizon_days=180)